# Distributed Shared-Codebook Experiment

This notebook defines the federated/shared-codebook experiment without a CLI wrapper. The default cells use deterministic tensors through `NeuralZipModel` so the experiment can run on CPU. Real checkpoint storage, download, compression sweeps, and upload are centralized in `1-model-hubs.ipynb`.


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "experiment_utils.py").exists():
    REPO_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "experiment_utils.py").exists():
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = Path("..").resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXAMPLES_DIR = REPO_ROOT / "examples"
OUTPUT_DIR = EXAMPLES_DIR / "outputs"

import pandas as pd
import torch

from experiment_utils import global_huffman_exponent_bits, neuralzip_v2_exponent_bits
from neuralzip_model import NeuralZipModel

try:
    from IPython.display import display
except ImportError:
    display = print


In [2]:
torch.manual_seed(17)

client_models = []
for client_id, scale in enumerate([0.5, 1.0, 2.0]):
    tensors = [
        (f"client_{client_id}.block0.weight", torch.randn(64, 64, dtype=torch.float32) * scale),
        (f"client_{client_id}.block1.weight", torch.randn(64, 32, dtype=torch.float32) * (scale / 2)),
        (f"client_{client_id}.head.bias", torch.randn(32, dtype=torch.float32) * 0.01),
    ]
    client_models.append(NeuralZipModel(tensors))

client_rows = []
for client_id, model in enumerate(client_models):
    named_exponents = model.named_exponents()
    layer_exponents = [exps for _, exps in named_exponents]
    v2_bits, v2_stats = neuralzip_v2_exponent_bits(model, named_exponents)
    raw_bits = sum(exps.size for exps in layer_exponents) * 8
    client_rows.append(
        {
            "client": client_id,
            "raw_exponent_bits": int(raw_bits),
            "separate_global_huffman_bits": int(global_huffman_exponent_bits(model, layer_exponents)),
            "separate_neuralzip_v2_bits": int(v2_bits),
            "neuralzip_v2_clusters": int(v2_stats["clusters"]),
        }
    )

client_table = pd.DataFrame(client_rows)
display(client_table)

,client,raw_exponent_bits,separate_global_huffman_bits,separate_neuralzip_v2_bits,neuralzip_v2_clusters
0,0,49408,17600,17824,3
1,1,49408,17320,17608,2
2,2,49408,17448,17640,3


In [3]:
federated_tensors = []
for model in client_models:
    federated_tensors.extend(list(model.iter_tensors()))

federated_model = NeuralZipModel(federated_tensors)
federated_named_exponents = federated_model.named_exponents()
federated_layer_exponents = [exps for _, exps in federated_named_exponents]
federated_v2_bits, federated_v2_stats = neuralzip_v2_exponent_bits(
    federated_model,
    federated_named_exponents,
)

communication_table = pd.DataFrame([
    {
        "strategy": "separate client codebooks",
        "scope": "sum of client exponent streams",
        "bits": int(client_table["separate_global_huffman_bits"].sum()),
    },
    {
        "strategy": "shared global Huffman codebook",
        "scope": "concatenated federated exponent stream",
        "bits": int(global_huffman_exponent_bits(federated_model, federated_layer_exponents)),
    },
    {
        "strategy": "shared NeuralZip v2 families",
        "scope": "concatenated federated exponent stream",
        "bits": int(federated_v2_bits),
    },
])
communication_table["ratio_vs_separate_huffman"] = (
    communication_table["bits"] / communication_table.loc[0, "bits"]
)
display(communication_table)

federated_v2_stats

,strategy,scope,bits,ratio_vs_separate_huffman
0,separate client codebooks,sum of client exponent streams,52368,1.000000
1,shared global Huffman codebook,concatenated federated exponent stream,55160,1.053315
2,shared NeuralZip v2 families,concatenated federated exponent stream,52240,0.997556


{'clusters': 6, 'families': 2, 'orders': {1: 6}}

In [4]:
model_hub_pipeline = pd.DataFrame([
    {
        "notebook": "1-model-hubs.ipynb",
        "scope": "model storing, checkpoint download, compression sweep, and upload",
        "reason": "the distributed notebook only studies shared-codebook behavior on controlled tensors",
    }
])
display(model_hub_pipeline)


,notebook,scope,reason
0,1-model-hubs.ipynb,"model storing, checkpoint download, compressio...",the distributed notebook only studies shared-c...
